# 論文 docx → J-STAGE XML 変換（試作版・Gradio の 1 画面）

**ツールの版: 0.26.1**（試用版。正式版は 1.0 から）

試作版です。確かめること: docx と PDF（10〜30 MB）をアップロードできるか、変換中に進捗が出るか、ZIP をダウンロードできるか、
API モードでシークレットを読めるか、準備にかかる時間。
上から順に、各セルの ▶ を押します（1. 準備が終わってから 2. 画面を開く）。

説明書: [使い方・結果の見方・できないこと](https://github.com/thjpme/jstgxmlcng_JASME/blob/main/docs/USAGE.md)　[AI の API キーと手動モード](https://github.com/thjpme/jstgxmlcng_JASME/blob/main/docs/USAGE_AI.md)　ソースコード: https://github.com/thjpme/jstgxmlcng_JASME

In [ ]:
#@title 1. 準備（最初に 1 回だけ。1 分ほどかかります）
#@markdown **CONVERT_EMF**: Word の図形（EMF/WMF）の図を変換するために、LibreOffice を裏でインストールします（通常はチェックのまま。数分かかりますが、次のセルは待たずに進めます）。
CONVERT_EMF = True  #@param {type:"boolean"}
#@markdown **USE_DEV_VERSION**: 開発者向け。開発中の版を試すときだけチェックします（ブランチ名を尋ねます）。
USE_DEV_VERSION = False  #@param {type:"boolean"}

import shutil, subprocess, sys
# このランタイムで一度ツールを使っていると、古い版がメモリに残ったままになる（入れ直しても反映されない）
loaded_before = "jasme_jats" in sys.modules
BRANCH = "main"
if USE_DEV_VERSION:
    BRANCH = input("開発版のブランチ名を入力して Enter を押してください: ").strip()
    if not BRANCH:
        raise SystemExit("ブランチ名が入力されていません。開発版を使わないときは、チェックを外して実行してください。")
url = f"git+https://github.com/thjpme/jstgxmlcng_JASME.git@{BRANCH}"
for args in (["jasme-jats[ai,images,colab,gui] @ " + url],                          # ツールと必要なライブラリ
             ["--force-reinstall", "--no-deps", "jasme-jats @ " + url]):  # ツール本体は毎回最新に入れ直す
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit("ツールのインストールに失敗しました。" + ("ブランチ名を確認してください。" if USE_DEV_VERSION else
                                                                  "しばらくしてからもう一度実行してください。"))
if loaded_before:
    raise SystemExit("ツールを最新の版に入れ替えましたが、このランタイムには前の版が読み込まれたままです。"
                     "メニューの「ランタイム」→「セッションを再起動」（表示によっては「ランタイムを再起動」）を選んでから、"
                     "1. 準備からもう一度実行してください。")
# LibreOffice はインストールに数分かかるので裏で入れる（変換のときに終わるのを待つ）
LO_LOG = "/content/libreoffice_install.log"
if CONVERT_EMF and not (shutil.which("soffice") or shutil.which("libreoffice")):
    if globals().get("LO_PROC") is None or LO_PROC.poll() is not None:
        LO_PROC = subprocess.Popen("apt-get -qq update && (apt-get -qq install -y --no-install-recommends "
                                   "libreoffice-draw-nogui || apt-get -qq install -y libreoffice-draw)",
                                   shell=True, stdout=open(LO_LOG, "w"), stderr=subprocess.STDOUT)
# 版の表示は別のプロセスで行う（ここで読み込むと、次に手順 1 を実行したときに「前の版が残っている」扱いになるため）
ver = subprocess.run([sys.executable, "-c", "from jasme_jats.pipeline import tool_version; print(tool_version())"],
                     capture_output=True, text=True).stdout.strip()
print("準備ができました。ツールの版:", ver)
if USE_DEV_VERSION:
    print("※ 開発版（ブランチ", BRANCH, "）を使っています。")
elif ver and not ver.startswith("0.26.1"):
    print("※ このノートブックの説明は版 0.26.1 のものです。ノートブックを開き直してください。")
if globals().get("LO_PROC") is not None and LO_PROC.poll() is None:
    print("LibreOffice を裏でインストールしています（数分）。インストールは裏で続くので、このまま次のセルに進んでください"
          "（変換は、インストールが終わってから始まります）。")

In [ ]:
#@title 2. 画面を開く
from jasme_jats.app import launch
launch()